### check libs & download model weights

In [1]:
!uv run env_check.py # !python3.12 env_check.py

Python: 3.12.3
Torch: 2.15.0+cu132
dllm: 0.1.0
lm_eval: 0.4.9.1
transformers: 4.57.0
trl: 0.24.0
CUDA: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
BF16: True


In [2]:
# hf auth login
!hf auth whoami

user:  gblssroman


In [3]:
from huggingface_hub import snapshot_download
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

model_paths = {
    "tiny": snapshot_download(
        repo_id="dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1",
        revision="c8d24a3f4adaeef46881b450e1bf7d1005203bd7",
        max_workers=16,
    ),
    "dream": snapshot_download(
        repo_id="Dream-org/Dream-v0-Instruct-7B",
        revision="05334cb9faaf763692dcf9d8737c642be2b2a6ae",
        max_workers=16,
    ),
}

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

---

### test models

In [4]:
import dllm
from dllm.core.samplers.mdlm import MDLMSampler, MDLMSamplerConfig
from dllm.pipelines.dream import DreamSampler, DreamSamplerConfig
import torch

def load_model(name):
    model = dllm.utils.get_model(
        model_name_or_path=model_paths[name],
        dtype="bfloat16",
        attn_implementation="eager" if name == "tiny" else "sdpa",
    ).eval()
    model.requires_grad_(False)

    tokenizer = dllm.utils.get_tokenizer(
        model_name_or_path=model_paths[name],
    )

    if name == "tiny":
        sampler = MDLMSampler(model=model, tokenizer=tokenizer)
        config = MDLMSamplerConfig(
            max_new_tokens=64,
            block_size=64,
            steps=64,
            temperature=0.0,
        )
    else:
        sampler = DreamSampler(model=model, tokenizer=tokenizer)
        config = DreamSamplerConfig(
            max_new_tokens=64,
            steps=64,
            temperature=0.0,
            alg="entropy",
        )

    return model, tokenizer, sampler, config


def run_agents(tokenizer, sampler, config):
    conversations = [
        [{"role": "user", "content": "Compute 3 + 4. Return only the answer."}],
        [{"role": "user", "content": "Compute 5 * 6. Return only the answer."}],
    ]

    inputs = tokenizer.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
    )

    with torch.inference_mode():
        outputs = sampler.sample(inputs, config, return_dict=True)

    answers = dllm.utils.sample_trim(
        tokenizer, outputs.sequences.tolist(), inputs,
    )
    for agent_id, answer in enumerate(answers):
        print(f"Agent {agent_id}: {answer}")

    return outputs

In [5]:
model, tokenizer, sampler, config = load_model("tiny")
res = run_agents(tokenizer, sampler, config) # 7, 30

res

/home/roman/Desktop/hidden-messages-diffusion-agents-math/.venv/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Agent 0: 7
Agent 1: 30


BaseSamplerOutput(sequences=tensor([[151644,    872,    198,  46254,    220,     18,    488,    220,     19,
             13,   3411,   1172,    279,   4226,     13, 151645,    198, 151644,
          77091,    198, 151667,    271, 151668,    271,     22, 151645,    198,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643],
        [151644,    872,    198,  46254,    220,     20,    353,    220,     21,
             13,   3411,   1172,    279,   4226,     13, 151645,    198, 151644,
          77091

In [6]:
del model, tokenizer, sampler, config, res

import gc
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [7]:
model, tokenizer, sampler, config = load_model("dream")
res = run_agents(tokenizer, sampler, config)

res

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Agent 0: 7
Agent 1: 30


BaseSamplerOutput(sequences=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  46254,    220,     18,    488,
            220,     19,     13,   3411,   1172,    279,   4226,     13, 151645,
            198, 151644,  77091,    198,     22, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643],
        [151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644

---

In [8]:
del model, tokenizer, sampler, config, res

import gc
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
STEPS = 32

def load_model(name):
    model = dllm.utils.get_model(
        model_name_or_path=model_paths[name],
        dtype="bfloat16",
        attn_implementation="eager" if name == "tiny" else "sdpa",
    ).eval()
    model.requires_grad_(False)

    tokenizer = dllm.utils.get_tokenizer(
        model_name_or_path=model_paths[name],
    )

    if name == "tiny":
        sampler = MDLMSampler(model=model, tokenizer=tokenizer)
        config = MDLMSamplerConfig(
            max_new_tokens=256,
            block_size=64,
            steps=STEPS,
            temperature=0.1,
        )
    else:
        sampler = DreamSampler(model=model, tokenizer=tokenizer)
        config = DreamSamplerConfig(
            max_new_tokens=256,
            steps=STEPS,
            temperature=0.1,
            alg="entropy",
        )

    return model, tokenizer, sampler, config

def run_agents(tokenizer, sampler, config):
    conversations = [
        [{"role": "user", "content": "Расскажи о том, как работает диффузия в ЛЛМ"}],
        [{"role": "user", "content": "Расскажи о том, как в принципе работает диффузия в DL"}],
    ]

    inputs = tokenizer.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
    )

    with torch.inference_mode():
        outputs = sampler.sample(inputs, config, return_dict=True)

    answers = dllm.utils.sample_trim(
        tokenizer, outputs.sequences.tolist(), inputs,
    )
    for agent_id, answer in enumerate(answers):
        print(f"Agent {agent_id}: {answer}")

    return outputs

model, tokenizer, sampler, config = load_model("tiny")
run_agents(tokenizer, sampler, config)

Agent 0: Диффузия (ЛД) - это процесс, в котором котором вещества перемешляются в жидкость, ведяя их их вперед в жидкость. В процессе диффузия происходит следующее образом:

1. В жидкости вещества перемешляются, ведяя их их вперед в жидкость.
2. В жидкости вещества перемешляются в жидкости, ведя их их вперед в жидкость.
3. В жидкости вещества перемешляются в жидкости, ведя их их вперед в жидкость.
4. В жидкости вещества перемешляются в жидкости ведя их их вперед в жидкость.
5. В жидкости вещества перемешляются в жидкости, ведяя их их вперед в жидкости.
6. В жидкости вещества перемешляются в жидкости, ведяя их вперед в жидкости.

Таким образом, диффузия (ЛД) - это процесс, в в которых вещества перемешляются в жидкости.
Agent 1: Диффузия - это процесс, при котором вещества перемешются в среде, изменяяя их распредел. В процессе, диффузия происходит из-за фактора, таких как давления, темперность и концентра вещества в среде.

Дляфузия - это процесс, при котором вещества перемешаются в среде

BaseSamplerOutput(sequences=tensor([[151644,    872,    198,  33504,  28400, 143847,   1802,   8215, 128856,
             11,  51670, 136533,   7796,  61709,  19355, 130272,  39131,   5805,
         124805,  63914,  37991, 151645,    198, 151644,  77091,    198, 151667,
            271, 151668,    271,  24110,  61709,  19355, 130272,  39131,    320,
          63914,  24110,      8,    481,  67879, 129644,     11,   5805, 133669,
         133669, 136789,   1478,  70801,  45967,  13695, 125484,   5805, 141997,
          96647,     11,   5805,  13103,   4235,   4235, 128488, 128488,   5805,
         143534,   5805, 141997,  96647,     13,  22933, 129644,   1504,   7796,
          61709,  19355, 130272,  39131, 136830,  92029,  71019,  49406, 132256,
           1447,     16,     13,  22933, 141997,  84764, 136789,   1478,  70801,
          45967,  13695, 125484,     11,   5805,  13103,   4235,   4235, 128488,
         128488,   5805, 143534,   5805, 141997,  96647,    624,     17,     13,
